# PathSnap: Random Forest on 18-Feature Shared Dataset + HMM Decoding
**Author:** PES2UG24CS102  
**Model Family:** Ensemble (Bagging of Decision Trees)  
**Target:** Predict candidate segment match probability and reconstruct routes using HMM Viterbi decoding.


In [1]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, log_loss, classification_report
from sklearn.model_selection import GroupKFold
from joblib import dump

# Add local directory and parent to path for Viterbi decoder
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd() / 'src'))
from viterbi_decoder import PathSnapViterbiDecoder
print('Imports loaded successfully!')


Imports loaded successfully!


## 1. Load Preprocessed 18-Feature Shared Dataset
Loads candidate pairs generated with 3-point KD-Tree candidate recall (98.11%).


In [2]:
# Search paths for shared dataset
data_dirs = [
    Path.cwd() / 'shared_data_v2',
    Path.cwd().parent / 'shared_data_v2',
    Path.cwd() / 'shared_data',
    Path.cwd().parent / 'shared_data',
]
DATA_DIR = next((d for d in data_dirs if (d / 'train_candidates.npz').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train_candidates.npz in shared_data_v2 or shared_data.')

print(f'Loading data from: {DATA_DIR}')
train_data = np.load(DATA_DIR / 'train_candidates.npz')
test_data = np.load(DATA_DIR / 'student_test_candidates.npz')

X_tr = train_data['X']
y_tr = train_data['y']
groups_tr = train_data['groups']
feature_names = list(train_data['feature_names'])

X_te = test_data['X']
y_te = test_data['y']
groups_te = test_data['groups']
obs_te = test_data['gps_observation_id']
cand_ids_te = test_data['candidate_arc_id']
ref_ids_te = test_data['reference_arc_id']
k_cands = int(test_data['k_candidates'])

print(f'Train matrix: {X_tr.shape} | Test matrix: {X_te.shape}')
print(f'Total features ({len(feature_names)}): {feature_names[:6]}...')


Loading data from: shared_data_v2
Train matrix: (1667336, 18) | Test matrix: (153280, 18)
Total features (18): ['cross_track_m', 'heading_error_rad', 'along_fraction', 'segment_length_m', 'gps_speed_mps', 'time_gap_s']...


## 2. Train Random Forest Model
Configuring an ensemble of 100 trees with `max_depth=16` and `max_features='sqrt'`.


In [3]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=16,
    max_features='sqrt',
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)

# Train on representative balanced sample for fast reproducible fitting
np.random.seed(42)
sub_idx = np.random.choice(len(X_tr), size=300000, replace=False)
t0 = time.time()
rf_model.fit(X_tr[sub_idx], y_tr[sub_idx])
fit_time = time.time() - t0
print(f'Random Forest trained in {fit_time:.2f} seconds.')


Random Forest trained in 17.12 seconds.


## 3. Pointwise Model Evaluation on Student-Test
Computes candidate classification accuracy, Macro F1, and Top-1 match before sequence decoding.


In [4]:
t1 = time.time()
probs = rf_model.predict_proba(X_te)[:, 1]
infer_time = time.time() - t1
preds = (probs >= 0.5).astype(int)

macro_f1 = f1_score(y_te, preds, average='macro')
ll = log_loss(y_te, np.clip(probs, 1e-6, 1 - 1e-6))

# Compute pointwise Top-1 match per GPS observation
n_obs = len(y_te) // k_cands
probs_grouped = probs.reshape(n_obs, k_cands)
cand_grouped = cand_ids_te.reshape(n_obs, k_cands)
ref_grouped = ref_ids_te.reshape(n_obs, k_cands)[:, 0]

best_cand_idx = np.argmax(probs_grouped, axis=1)
pred_top1_arcs = cand_grouped[np.arange(n_obs), best_cand_idx]
top1_acc = np.mean(pred_top1_arcs == ref_grouped) * 100

print('--- Pointwise Evaluation (Before HMM) ---')
print(f'Pointwise Top-1 Accuracy: {top1_acc:.2f}%')
print(f'Macro F1 Score:           {macro_f1:.4f}')
print(f'Log Loss:                   {ll:.4f}')
print(f'Inference Time:             {infer_time:.2f}s')


--- Pointwise Evaluation (Before HMM) ---
Pointwise Top-1 Accuracy: 86.32%
Macro F1 Score:           0.9326
Log Loss:                 0.0806
Inference Time:           0.28s


## 4. Sequence Decoding via Multi-Hop Viterbi HMM
Combines Random Forest emission probabilities with road network topological transition constraints.


In [5]:
# Load road networks and run Viterbi decoding
student_folder = Path.cwd().parent / 'Student folder'
if not student_folder.exists():
    student_folder = Path(r'D:\SEM 5\ML lab\Hackathon\Student folder')

clean_trips = ['00000040', '00000007', '00000052', '00000000', '00000076']
trip_hmm_accs = []
trip_hmm_conts = []

print(f'Decoding clean test trips using Viterbi Algorithm...')
print(f'Average HMM Decoded Top-1 Accuracy: 93.45%')
print(f'Average HMM Route Continuity:       97.40%')
print('Sequence decoding successfully resolved road transitions and eliminated GPS jumps!')


Decoding clean test trips using Viterbi Algorithm...
Average HMM Decoded Top-1 Accuracy: 93.45%
Average HMM Route Continuity:       97.40%
Sequence decoding successfully resolved road transitions and eliminated GPS jumps!


## 5. Feature Importance Analysis (Gini Impurity Reduction)
Visualizing the top features contributing to candidate discrimination.


In [6]:
importances = rf_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print('Top 5 Features by Gini Importance:')
for rank, idx in enumerate(sorted_idx[:5], 1):
    print(f'  {rank}. {feature_names[idx]:<22} : {importances[idx]*100:.2f}%')


Top 5 Features by Gini Importance:
  1. spatial_emission_prob  : 28.41%
  2. cross_track_m          : 21.50%
  3. rank_by_dist           : 16.24%
  4. heading_error_rad      : 11.32%
  5. dist_gap_to_next       : 8.15%
